# Day 3 practice — Order forms and plating standards

**Read first:** [05_theory_pydantic_models.md](05_theory_pydantic_models.md)

Today's headline demo: you will watch a `response_model` refuse to leak a password hash that the
handler explicitly returned. That single behaviour is worth the whole day.

In [ ]:
import json
from datetime import date, datetime
from decimal import Decimal

from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel, ConfigDict, Field, ValidationError, field_validator

def show(r, label=""):
    print(f"{label:<38} {r.status_code}  {r.text[:160]}")

print("ready")

---

## Part 1 — A model is a declaration, not code

In [ ]:
class CityIn(BaseModel):
    name: str
    latitude: float
    longitude: float
    country: str = "NL"
    population: int | None = None

c = CityIn(name="Utrecht", latitude=52.09, longitude=5.12)

print("attribute access :", c.name, "|", c.latitude)
print("default filled in:", c.country)
print("model_dump()     :", c.model_dump())
print("model_dump_json():", c.model_dump_json())

> ⚠️ **v1 vs v2.** Older code and most blog posts call `.dict()`, `.json()`, and `parse_obj()`.
> Those are **Pydantic v1**. You are on **v2**: `model_dump()`, `model_dump_json()`,
> `model_validate()`. If a snippet from the internet calls `.dict()`, it was written for a version
> you are not running.

### Coercion: strings become numbers, when they can

In [ ]:
coerced = CityIn(name="Utrecht", latitude="52.09", longitude="5.12", population="360000")
print("latitude  :", coerced.latitude,   type(coerced.latitude).__name__)
print("population:", coerced.population, type(coerced.population).__name__)

try:
    CityIn(name="Utrecht", latitude="fifty-two", longitude=5.12)
except ValidationError as exc:
    print()
    print("but nonsense is refused:")
    print(exc)

### 🔮 Predict — required vs optional

Four spellings that look similar. For each, will the model accept a **missing** key? An explicit
**`null`**?

| Field | key may be missing? | value may be null? |
|---|---|---|
| `a: int` | ? | ? |
| `b: int = 0` | ? | ? |
| `c: int \| None` | ? | ? |
| `d: int \| None = None` | ? | ? |

In [ ]:
class Matrix(BaseModel):
    a: int
    b: int = 0
    c: int | None
    d: int | None = None

def try_it(payload, label):
    try:
        Matrix.model_validate(payload)
        print(f"  ✅ {label}")
    except ValidationError as exc:
        missing = [e["loc"][0] for e in exc.errors()]
        print(f"  ❌ {label:<38} failed on {missing}")

print("everything supplied:")
try_it({"a": 1, "b": 2, "c": 3, "d": 4}, "all four")

print()
print("one key omitted at a time (base = a,b,c,d all present):")
base = {"a": 1, "b": 2, "c": 3, "d": 4}
for key in "abcd":
    payload = {k: v for k, v in base.items() if k != key}
    try_it(payload, f"without '{key}'")

print()
print("explicit null, one at a time:")
for key in "abcd":
    payload = dict(base, **{key: None})
    try_it(payload, f"'{key}' = null")

**The type controls what values are allowed. The default controls whether the key may be
missing.** They are independent knobs, and conflating them is the most common Pydantic confusion.

`c: int | None` is the interesting one: you **must** send the key, and `null` is a valid answer.
That's how you say "you have to tell me, and 'nothing' counts as telling me".

---

## Part 2 — Models as request bodies

In [ ]:
app = FastAPI()
client = TestClient(app)

@app.post("/cities", status_code=201)
def create_city(city: CityIn):          # a Pydantic model -> read from the BODY
    return {"created": city.name, "country": city.country}

show(client.post("/cities", json={"name": "Zwolle", "latitude": 52.5, "longitude": 6.1}), "valid")
show(client.post("/cities", json={"name": "Zwolle"}), "missing two fields")

### Every error at once, each with a breadcrumb

In [ ]:
bad = client.post("/cities", json={"name": "Zwolle", "latitude": "fifty-two"}).json()
print(json.dumps(bad, indent=2))
print()
for e in bad["detail"]:
    print(f"  {'.'.join(str(p) for p in e['loc']):<22} {e['type']:<16} {e['msg']}")

Two problems reported in **one** response. The caller fixes both in a single round trip instead
of discovering them one at a time over five requests.

### The complete parameter-source rule

| Parameter | Comes from |
|---|---|
| name appears in the path | **Path** |
| scalar, not in the path | **Query** |
| Pydantic model | **Body** |
| `Depends(...)` | a dependency (Day 4) |

All three at once, in one signature:

In [ ]:
@app.put("/cities/{name}")
def replace_city(name: str, city: CityIn, notify: bool = False):
    #             ^ PATH        ^ BODY        ^ QUERY
    return {"path_name": name, "body_name": city.name, "notify": notify}

show(client.put("/cities/Zwolle?notify=true",
                json={"name": "Zwolle", "latitude": 52.5, "longitude": 6.1}), "path+body+query")

---

## Part 3 — `Field` and `field_validator`

In [ ]:
class StrictCity(BaseModel):
    name: str = Field(min_length=1, max_length=100, description="City name")
    latitude: float = Field(ge=-90, le=90)
    longitude: float = Field(ge=-180, le=180)
    population: int | None = Field(default=None, gt=0)

    @field_validator("name")
    @classmethod
    def tidy_name(cls, v: str) -> str:
        v = v.strip()
        if not v:
            raise ValueError("name cannot be blank")
        return v.title()               # the RETURN VALUE REPLACES the field

print("normalised:", StrictCity(name="   uTRECHT  ", latitude=52.09, longitude=5.12).name)

for payload, label in [
    ({"name": "X", "latitude": 500, "longitude": 5}, "latitude 500"),
    ({"name": "   ", "latitude": 52, "longitude": 5}, "blank name"),
    ({"name": "X", "latitude": 52, "longitude": 5, "population": 0}, "population 0"),
]:
    try:
        StrictCity.model_validate(payload)
        print(f"  ✅ {label}")
    except ValidationError as exc:
        print(f"  ❌ {label:<16} {exc.errors()[0]['msg']}")

A validator is also a **normaliser**: `"   uTRECHT  "` became `"Utrecht"` before any handler saw
it. You clean data once, at the door, instead of in every function that touches it.

Note it raises `ValueError`, **not** `HTTPException`. Pydantic knows nothing about HTTP; FastAPI
translates. Keeping models HTTP-free is what lets you reuse them in a CLI or a background job.

---

## Part 4 — 🚨 `response_model`: the demo that matters

Here is a realistic disaster. A handler fetches a database row and returns it.

In [ ]:
FAKE_USER_ROW = {
    "id": 42,
    "email": "student@example.com",
    "password_hash": "$2b$12$SUPERSECRETHASHDONOTLEAK",   # <- added by a colleague last week
    "internal_notes": "flagged for review",
}

leaky = FastAPI()

@leaky.get("/users/{user_id}")              # NO response_model
def read_user_leaky(user_id: int):
    return FAKE_USER_ROW

print("WITHOUT response_model:")
print(json.dumps(TestClient(leaky).get("/users/42").json(), indent=2))

**The password hash is on the public internet.** Nothing in the handler changed — someone added
a column, and every endpoint that returned `dict(row)` started publishing it. No test failed. No
error was logged.

Now the same handler, with a plating standard:

In [ ]:
class UserOut(BaseModel):
    id: int
    email: str
    # password_hash is NOT declared. That is the entire security control.

safe = FastAPI()

@safe.get("/users/{user_id}", response_model=UserOut)
def read_user_safe(user_id: int):
    return FAKE_USER_ROW                    # still returning EVERYTHING

body = TestClient(safe).get("/users/42").json()
print("WITH response_model:")
print(json.dumps(body, indent=2))
print()
print("password_hash present?", "password_hash" in body)
print("internal_notes present?", "internal_notes" in body)

> 🎯 **Remember this** — `response_model` is an **allow-list**, and allow-lists fail closed. A
> field nobody declared cannot leak, no matter what the handler returns.

### The test that would have caught it

Day 5 covers this properly, but note the shape now: here, **exact key equality is right**, because
the complete key set *is* the promise.

```python
assert set(response.json()) == {"id", "email"}
```

---

## Part 5 — Conversion at the boundary

Day 1 ended with `json.dumps` refusing a `Decimal` and a `date`. Your Module 2 `marts` tables are
full of `NUMERIC` columns, which arrive in Python as `Decimal`. Watch the boundary fix it.

In [ ]:
row_from_postgres = {
    "city": "Utrecht",
    "obs_date": date(2026, 9, 1),
    "temp_max_c": Decimal("21.40"),
    "loaded_at": datetime(2026, 9, 2, 5, 0, 0),
}

print("Plain json.dumps:")
try:
    json.dumps(row_from_postgres)
except TypeError as exc:
    print("   TypeError:", exc)

class WeatherOut(BaseModel):
    city: str
    obs_date: date
    temp_max_c: float          # <- declaring float does the Decimal conversion

conv = FastAPI()

@conv.get("/weather", response_model=WeatherOut)
def read_weather():
    return row_from_postgres

print()
print("Through a response_model:")
print(json.dumps(TestClient(conv).get("/weather").json(), indent=2))

`Decimal("21.40")` became `21.4`, the `date` became `"2026-09-01"`, and `loaded_at` was dropped
because it wasn't declared. **No handler had to know.**

### Reading ORM objects: `from_attributes`

A SQLAlchemy row is an object with attributes, not a dict.

In [ ]:
class FakeRow:                     # stands in for a SQLAlchemy row
    def __init__(self):
        self.city = "Utrecht"
        self.obs_date = date(2026, 9, 1)
        self.temp_max_c = Decimal("21.40")

try:
    WeatherOut.model_validate(FakeRow())
except ValidationError as exc:
    print("without from_attributes:", exc.errors()[0]["msg"])

class WeatherOutORM(BaseModel):
    model_config = ConfigDict(from_attributes=True)      # v2 spelling (v1 used `class Config`)
    city: str
    obs_date: date
    temp_max_c: float

print()
print("with from_attributes   :", WeatherOutORM.model_validate(FakeRow()))

That one line is why the project's `models.py` sets `from_attributes=True` on every output model:
handlers can return database rows directly.

---

## Part 6 — Nested models and the `loc` trail

In [ ]:
class Location(BaseModel):
    lat: float = Field(ge=-90, le=90)
    lon: float = Field(ge=-180, le=180)

class CityNested(BaseModel):
    name: str
    location: Location
    tags: list[str] = []

nested = FastAPI()

@nested.post("/cities")
def create(city: CityNested):
    return {"ok": city.name}

nc = TestClient(nested)
show(nc.post("/cities", json={"name": "Utrecht",
                              "location": {"lat": 52.09, "lon": 5.12}}), "valid nested")

err = nc.post("/cities", json={"name": "Utrecht", "location": {"lat": 999, "lon": 5.12}}).json()
print()
print("loc for a nested failure:", err["detail"][0]["loc"])

In [ ]:
# A list of models: loc gains an INDEX, so you know WHICH item was wrong.
class Batch(BaseModel):
    cities: list[CityNested]

@nested.post("/batch")
def create_batch(batch: Batch):
    return {"count": len(batch.cities)}

payload = {"cities": [
    {"name": "A", "location": {"lat": 52.0, "lon": 5.0}},
    {"name": "B", "location": {"lat": 52.0, "lon": 5.0}},
    {"name": "C", "location": {"lat": 999, "lon": 5.0}},      # index 2 is bad
]}
err = nc.post("/batch", json=payload).json()
for e in err["detail"]:
    print("loc:", e["loc"], "->", e["msg"])

`["body", "cities", 2, "location", "lat"]` — outside in, with the list index. When a caller sends
you 500 rows and one is malformed, this tells them exactly which.

---

## Exercises

### Exercise 1 — In/Out pair (⭐)

Write `ObservationIn` (what a caller may send: `city`, `obs_date`, `temp_max_c`, `temp_min_c`) and
`ObservationOut` (what you send back: everything above **plus** a server-assigned `id` and
`loaded_at`). A caller must **not** be able to supply `id`.

In [ ]:
# Your code here


<details>
<summary>💡 Solution</summary>

```python
class ObservationBase(BaseModel):
    city: str = Field(min_length=1)
    obs_date: date
    temp_max_c: float | None = None
    temp_min_c: float | None = None

class ObservationIn(ObservationBase):
    pass

class ObservationOut(ObservationBase):
    model_config = ConfigDict(from_attributes=True)
    id: int
    loaded_at: datetime
```

The asymmetry is the point. `id` and `loaded_at` exist only on the way **out**, so a caller cannot
choose their own primary key. Sharing the common fields through a base class means a new field is
added in one place, not two.

Prove a caller can't set `id`:

```python
o = ObservationIn.model_validate(
    {"city": "X", "obs_date": "2026-09-01", "id": 9999}
)
print(hasattr(o, "id"))   # False - the extra key was ignored, not accepted
```
</details>

### Exercise 2 — Validate a date range (⭐⭐)

Write a `DateRange` model with `from_date` and `to_date` where `to_date` must not be **before**
`from_date`. (Hint: a `field_validator` only sees one field. Look up `model_validator(mode="after")`,
which sees the whole object.)

In [ ]:
# Your code here


<details>
<summary>💡 Solution</summary>

```python
from pydantic import model_validator

class DateRange(BaseModel):
    from_date: date
    to_date: date

    @model_validator(mode="after")
    def check_order(self):
        if self.to_date < self.from_date:
            raise ValueError("to_date must not be before from_date")
        return self          # model validators return the MODEL, not a value

print(DateRange(from_date=date(2026, 9, 1), to_date=date(2026, 9, 30)))

try:
    DateRange(from_date=date(2026, 9, 30), to_date=date(2026, 9, 1))
except ValidationError as exc:
    print(exc.errors()[0]["msg"])
```

`mode="after"` runs once every individual field has already been parsed and validated, so
`self.from_date` is guaranteed to be a real `date` by then. `mode="before"` would see the raw input
instead. And unlike a `field_validator`, this one returns `self`.
</details>

### Exercise 3 — Prove the allow-list (⭐⭐⭐)

Build an endpoint returning a dict with **six** keys, declare a `response_model` with only **three**,
and write assertions proving the other three are absent. Then add a *fourth* field to the model that
does **not** exist in the returned dict, and explain the error you get.

In [ ]:
# Your code here


<details>
<summary>💡 Solution</summary>

```python
ex3 = FastAPI()

ROW = {"id": 1, "email": "a@b.c", "name": "Ann",
       "password_hash": "x", "ssn": "y", "internal": "z"}

class Out3(BaseModel):
    id: int
    email: str
    name: str

@ex3.get("/u", response_model=Out3)
def u():
    return ROW

body = TestClient(ex3).get("/u").json()
assert set(body) == {"id", "email", "name"}
for leaked in ("password_hash", "ssn", "internal"):
    assert leaked not in body
print("nothing leaked:", body)
```

Now add a field the handler never returns:

```python
class Out4(Out3):
    phone: str          # not in ROW, and no default

@ex3.get("/u2", response_model=Out4)
def u2():
    return ROW

print(TestClient(ex3, raise_server_exceptions=False).get("/u2").status_code)   # 500
```

You get a **500**, not a 422 — and that is correct. A `422` means *the caller* sent something wrong.
Here the caller did nothing wrong: **we** promised a `phone` field and failed to produce it. That is
a server bug, and `response_model` caught it at the boundary instead of shipping a broken payload.

This is the fourth job of `response_model` from the theory doc: it validates *your own output*, so
this fails loudly in development rather than confusing a client six months later.
</details>

---

## ✅ Before you move on

- Which knob makes a field optional: the type, or the default?
- What are the four jobs of `response_model`?
- Why does a validator `raise ValueError` rather than `HTTPException`?
- What does `from_attributes=True` buy you?
- What does `loc: ["body", "cities", 2, "location", "lat"]` tell you?

Next: **[Day 4 theory](../day4-dependencies-and-db/07_theory_dependency_injection.md)** — the API
finally meets your Module 2 database.